### Welcome!
<style>
div.text_cell_render {
    background-color: transparent !important;
    border: none !important;
    padding: 0px !important;
}
</style>

## House keeping and load modules

In [ ]:
%%time 

# autoload external python modules if they changed
%load_ext autoreload
%autoreload 2
# add ../funcs to the current path
import sys, os
sys.path.append(os.path.join(os.getcwd(), "..")) 

# import modules
import warnings
import cartopy.crs as ccrs
import geoviews.feature as gf
import holoviews as hv
import hvplot.xarray
from holoviews import opts
import uxarray as ux
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
hv.extension("bokeh")
# hv.extension("matplotlib")

import geoviews as gv
import cartopy.crs as ccrs
import geopandas as gp

from DAmonitor.shapes import coast_lines, state_lines, county_lines
from DAmonitor.mpas import hslice_contour, vslice_contour

## load MPAS data

In [ ]:
%%time
grid_file="../data/samples/mpasjedi/invariant.nc"
bkg_file="../data/samples/mpasjedi/bkg.nc"
ana_file="../data/samples/mpasjedi/ana.nc"

uxds_a = ux.open_dataset(grid_file, ana_file)
uxds_b = ux.open_dataset(grid_file, bkg_file)

## make a horizontal temperature plot

In [ ]:
%%time

uxvar0 = uxds_a['theta'] - 273.15
uxvar = uxvar0
nt = 0  # time dimension
lev = 0  # vertical level
plot = hslice_contour(uxvar.isel(Time=nt, nVertLevels=lev), title=f'lev={lev}')
plot * coast_lines * state_lines

## setting up a subdomain box

In [ ]:
lon_center = -105.03
lat_center = 39.0
lon_incr = 5 # degree
lat_incr = 3 # degree
lon_bounds = (lon_center - lon_incr, lon_center + lon_incr)
lat_bounds = (lat_center - lat_incr, lat_center + lat_incr)

## plot temperature analysis increments at different levels

In [ ]:
%%time

var_name = "theta"
uxdiff0 = uxds_a[var_name] - uxds_b[var_name]
uxvar = uxdiff0

nt=0  # time dimension
plot_levels = [42]  # [0, 29, 42]  # [0, 19, 29, 39, 49, 58]

plots = []
for lev in plot_levels:
    tmp = hslice_contour(uxvar.isel(Time=nt, nVertLevels=lev), title=f'lev={lev}')  # for the whole domain    
    plots.append(tmp * coast_lines * state_lines)

# plots share one toolbar, which facilitates doing sync'ed zoom-in/out
# hv.Layout(plots).cols(1)

# each plot has its own toolbar, which facilitates controlling each plot individually
for p in plots:
   display(p)

## plot temperature increments zoomed into the Colorado domain

In [ ]:
%%time

var_name = "theta"
uxdiff0 = uxds_a[var_name] - uxds_b[var_name]

### subset to a small domain
uxdiff1 = uxdiff0.subset.bounding_box(lon_bounds, lat_bounds,)
uxvar = uxdiff1

nt=0  # time dimension
plot_levels = [42]  # [0, 29, 42]  # [0, 19, 29, 39, 49, 58]

plots = []
for lev in plot_levels:
    tmp = hslice_contour(uxvar.isel(Time=nt, nVertLevels=lev), title=f'lev={lev}', width=700, height=500)  # for the subdomain
    
    # overlay state_lines
    #plots.append(tmp * coast_lines * state_lines)  
    
    # overlay county lines, this takes longer time to render
    plots.append(tmp * coast_lines * county_lines.opts(xlim=(lon_bounds[0], lon_bounds[1]), ylim=(lat_bounds[0], lat_bounds[1])))

# plots share one toolbar, which facilitates doing sync'ed zoom-in/out
# hv.Layout(plots).cols(1)

# each plot has its own toolbar, which facilitates controlling each plot individually
for p in plots:
   display(p)

## plot a vertcial cross section of temperature increments

In [ ]:
%%time

var_name = "theta"
uxdiff0 = uxds_a[var_name] - uxds_b[var_name]
uxvar = uxdiff0

### each plot has its own toolbar

In [ ]:
%%time

tmp = vslice_contour(uxvar, lon=-85.77, clevels=10)
display(tmp)
tmp = vslice_contour(uxvar, lat=42.63, clevels=10)
display(tmp)

### save plots to files

In [ ]:
hv.save(tmp, 'vslice.png')

### plots share one toolbar

In [ ]:
%%time 

( vslice_contour(uxvar, lon=-85.77, clevels=10) +
vslice_contour(uxvar, lat=42.63, clevels=10) ).cols(1)